In [2]:
"""
============================================================
A2A COUNTER CLIENT
============================================================

Purpose
-------
Calls the Counter Agent using the official A2A SDK 1.x.

The client sends:

    "count to 10"

The Counter Agent streams:

    counted 1/10
    counted 2/10
    counted 3/10
    ...
    counted 10/10

Finally:

    Finished counting to 10.


============================================================
CLIENT WORKFLOW
============================================================

        Counter Client
              |
              |
              | 1. GET Agent Card
              v
        A2ACardResolver
              |
              v
          AgentCard
              |
              |
              | 2. create_client()
              v
          A2A Client
              |
              |
              | 3. message/send
              |
              | "count to 10"
              v
        Counter Agent
              |
              | streaming events
              v
       +------+------+
       |             |
       v             v
    artifact      status/task
    updates       updates
       |
       v
    1/10
    2/10
    3/10
     ...
    10/10
       |
       v
    completed


============================================================
A2A SDK 1.x
============================================================

A2A SDK 1.x uses protobuf messages.

Therefore:

    agent_card.model_dump()       ❌

Use:

    MessageToJson(agent_card)     ✅


The message structure is:

    SendMessageRequest
            |
            v
         Message
            |
            v
           Part
            |
            v
      "count to 10"


============================================================
JUPYTER
============================================================

Run:

    await main()

Do NOT use:

    asyncio.run(main())

because Jupyter already has an event loop.
"""

import uuid

import httpx

from google.protobuf.json_format import MessageToJson

from a2a.client import (
    A2ACardResolver,
    ClientConfig,
    create_client,
)

from a2a.types import (
    Message,
    Part,
    Role,
    SendMessageRequest,
)


# ============================================================
# COUNTER AGENT URL
# ============================================================

AGENT_URL = "http://localhost:9112"


# ============================================================
# PROTOBUF PRINT HELPER
# ============================================================
#
# A2A SDK 1.x objects are protobuf messages.
#
# Therefore we use:
#
#     MessageToJson()
#
# instead of:
#
#     model_dump()
#
# ============================================================

def print_proto(
    title,
    proto_msg,
):

    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)

    print(
        MessageToJson(
            proto_msg,
            indent=2,
        )
    )


# ============================================================
# MAIN CLIENT WORKFLOW
# ============================================================

async def main():

    print("=" * 60)
    print("A2A COUNTER CLIENT")
    print("=" * 60)


    # ========================================================
    # STEP 1
    # CREATE HTTP CLIENT
    # ========================================================
    #
    # All communication with the remote A2A agent happens
    # through this HTTP client.
    #
    # ========================================================

    async with httpx.AsyncClient() as httpx_client:


        # ====================================================
        # STEP 2
        # DISCOVER THE AGENT
        # ====================================================
        #
        # The client does not need to know every capability
        # of the Counter Agent beforehand.
        #
        # It asks the agent for its Agent Card.
        #
        # HTTP:
        #
        # GET
        # http://localhost:9102/
        # .well-known/agent-card.json
        #
        # ====================================================

        print("\nGetting Agent Card...")

        resolver = A2ACardResolver(
            httpx_client,
            AGENT_URL,
        )

        agent_card = await resolver.get_agent_card()


        # ----------------------------------------------------
        # Display basic information from Agent Card.
        # ----------------------------------------------------

        print(
            "\nAgent:",
            agent_card.name,
        )

        print_proto(
            "COUNTER AGENT CARD",
            agent_card,
        )


        # ====================================================
        # STEP 3
        # CREATE A2A CLIENT
        # ====================================================
        #
        # The Agent Card tells the client how the agent
        # describes itself.
        #
        # We now create an SDK client capable of communicating
        # with that agent.
        #
        # streaming=True because Counter sends progress
        # updates while it is working.
        #
        # ====================================================

        config = ClientConfig(
            httpx_client=httpx_client,
            streaming=True,
        )

        client = await create_client(
            agent_card,
            client_config=config,
        )


        # ====================================================
        # STEP 4
        # BUILD A2A MESSAGE
        # ====================================================
        #
        # User wants:
        #
        #     "count to 10"
        #
        # The A2A SDK expects:
        #
        # SendMessageRequest
        #       |
        #       +--> Message
        #                |
        #                +--> Part
        #                       |
        #                       +--> text
        #
        # ====================================================

        request = SendMessageRequest(

            message=Message(

                # Every message gets a unique ID.
                message_id=str(
                    uuid.uuid4()
                ),

                # This message is from the user.
                role=Role.ROLE_USER,

                # Message content.
                parts=[
                    Part(
                        text="count to 10"
                    )
                ],
            )
        )


        # ====================================================
        # STEP 5
        # SEND MESSAGE
        # ====================================================
        #
        # This represents the A2A:
        #
        #     message/send
        #
        # The important point:
        #
        # send_message() returns an ASYNC ITERATOR.
        #
        # Therefore we use:
        #
        #     async for event
        #
        # ====================================================

        print("\n")
        print("=" * 60)
        print("SENDING REQUEST")
        print("=" * 60)

        print(
            "Request: count to 10"
        )


        async for event in client.send_message(
            request
        ):


            # =================================================
            # STEP 6
            # IDENTIFY EVENT TYPE
            # =================================================
            #
            # A2A can return different event types.
            #
            # Examples:
            #
            #     message
            #     task
            #     status_update
            #     artifact_update
            #
            # =================================================
            print(event)
            kind = event.WhichOneof(
                "payload"
            )


            # =================================================
            # STEP 7
            # HANDLE MESSAGE
            # =================================================
            #
            # Example:
            #
            #     "Counting..."
            #
            # or:
            #
            #     "Finished counting to 10."
            #
            # =================================================

            if kind == "message":

                texts = [
                    part.text
                    for part in event.message.parts
                    if part.text
                ]

                if texts:

                    print(
                        "\nMessage:",
                        " ".join(texts),
                    )


            # =================================================
            # STEP 8
            # HANDLE ARTIFACT UPDATE
            # =================================================
            #
            # Counter Agent uses:
            #
            #     updater.add_artifact()
            #
            # to send progress.
            #
            # So the client receives updates such as:
            #
            #     counted 1/10
            #     counted 2/10
            #     counted 3/10
            #
            # =================================================

            elif kind == "artifact_update":

                print(
                    "\nProgress update:"
                )

                print(
                    event.artifact_update
                )


            # =================================================
            # STEP 9
            # HANDLE TASK
            # =================================================
            #
            # The client may receive the Task itself.
            #
            # This contains task-level information such as
            # task ID, context ID, history, artifacts, etc.
            #
            # =================================================

            elif kind == "task":

                print(
                    "\nTask:"
                )

                print(
                    event.task
                )


            # =================================================
            # STEP 10
            # HANDLE STATUS UPDATE
            # =================================================
            #
            # This can tell us about task lifecycle changes.
            #
            # For example:
            #
            #     working
            #     completed
            #     canceled
            #
            # =================================================

            elif kind == "status_update":

                print(
                    "\nStatus update:"
                )

                print(
                    event.status_update
                )


            # =================================================
            # STEP 11
            # PRINT RAW EVENT
            # =================================================
            #
            # This is extremely useful while learning A2A.
            #
            # It lets us see exactly what the SDK is sending
            # over the wire.
            #
            # Once you understand A2A, you can remove this.
            #
            # =================================================

            print_proto(
                f"RAW RESPONSE ({kind})",
                event,
            )


# ============================================================
# JUPYTER ENTRY POINT
# ============================================================
#
# Because Jupyter already runs an asyncio event loop:
#
#     await main()
#
# Do NOT use:
#
#     asyncio.run(main())
#
# ============================================================

await main()

A2A COUNTER CLIENT

Getting Agent Card...

Agent: counter

COUNTER AGENT CARD
{
  "name": "counter",
  "description": "Counts to a number, one step per second, reporting progress.",
  "supportedInterfaces": [
    {
      "url": "http://127.0.0.1:9112/",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "version": "1.0.0",
  "capabilities": {
    "streaming": true
  },
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/plain"
  ],
  "skills": [
    {
      "id": "count",
      "name": "Count slowly",
      "description": "Counts to N, emitting progress updates.",
      "tags": [
        "a2a",
        "learning-example"
      ],
      "examples": [
        "count to 10"
      ],
      "inputModes": [
        "text/plain"
      ],
      "outputModes": [
        "text/plain"
      ]
    }
  ]
}


SENDING REQUEST
Request: count to 10
task {
  id: "2383e116-093a-47f0-9a62-832379906878"
  context_id: "8e49b056-dba2-4fb0-a9b4-28663